# Recurrent HTGs: Stateful Nodes with Sequence Rollout

## Overview

A **recurrent node** carries internal **state** that persists across time steps in a sequence.
In CREST, a node is made recurrent by passing `recurrent=True` to the `Node` constructor.

Two additional elements are required for every recurrent node:

1. **initial_state(self, X)** — returns a dict of initial state values (typically `None` or
   zeros) before the first time step. CREST calls this automatically at the start of each rollout.

2. **rollout_axis** on `add_edge` — the tensor axis along which the graph iterates to produce
   a sequence. With `rollout_axis=1`, inputs of shape `(batch, time, features)` are stepped
   through one slice at a time along axis 1 (the time axis). CREST defaults to `rollout_axis=1`
   for all edges targeting a recurrent node if not specified explicitly.

**Recurrent edges** — self-loops or back-edges between nodes — carry a node's output state at
time *t* as an input at time *t+1*. The graph topology determines which states are shared and
in which direction.

This notebook demonstrates various recurrent topologies using simple arithmetic nodes so the mechanics are easy to follow.

In [ ]:
# Allow crest to be imported
%cd -q ..

In [ ]:
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph
from crest.model.Node import Node
import numpy as np
import tensorflow as tf

## Example 1: Two Mutually-Coupled Recurrent Nodes

`Add1` and `Add2` share state bidirectionally at every time step. Each node receives its own
previous state **plus** the other node's previous state, alongside the current time-step input:

- `Add1`: `s_1[t] = s_1[t-1] + s_2[t-1] + x_1[t]`
- `Add2`: `s_2[t] = s_1[t-1] + s_2[t-1] + x_2[t]`

Both self-loops (`add_1 → add_1`, `add_2 → add_2`) and cross-edges (`add_1 ↔ add_2`) are
needed. This bidirectional pattern is common in coupled Earth system components (e.g. ocean
exchanging state with atmosphere at every model time step).

In [ ]:
class Add1(Node):
    def __init__(self, name):
        def add(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            s_2 = X['s_2'] if X['s_2'] is not None else 0
            v = s_1 + s_2 + X['x_1']
            return {'s_1':  v}
        super().__init__(
            node=add,
            inputs={'x_1': np.ones((1, 3)), 's_2': np.ones((1,)), 's_1': np.ones((1,))},
            outputs={'s_1': np.ones((1,))},
            name=name,
            recurrent=True
        )

    def initial_state(self, X):
        return {'s_1': None}


class Add2(Node):
    def __init__(self, name):
        def add(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            s_2 = X['s_2'] if X['s_2'] is not None else 0
            v = s_1 + s_2 + X['x_2']
            return {'s_2':  v}
        super().__init__(
            node=add,
            inputs={'x_2': np.ones((1, 3)), 's_2': np.ones((1,)), 's_1': np.ones((1,))},
            outputs={'s_2': np.ones((1,))},
            name=name,
            recurrent=True
        )

    def initial_state(self, X):
        return {'s_2': None}

In [ ]:
htg = HierarchalTensorGraph(name='parent')
add_1 = Add1('add_1')
add_2 = Add2('add_2')

htg.add_edge('input', add_1, rollout_axis=1, features=['x_1'])
htg.add_edge('input', add_2, rollout_axis=1, features=['x_2'])
htg.add_edge(add_1, add_2, rollout_axis=1)
htg.add_edge(add_2, add_1, rollout_axis=1)
htg.add_edge(add_1, add_1, rollout_axis=1)
htg.add_edge(add_2, add_2, rollout_axis=1)
htg.add_edge(add_1, 'output')
htg.add_edge(add_2, 'output')

In [ ]:
htg.print_edge_labels()

In [ ]:
htg.draw()

## Example 2: Single Recurrent Node with Self-Loop

The simplest recurrent graph is a single node that feeds its own output back as input at the
next time step — a **self-loop**. Here, `Add1` computes a running cumulative sum:

`s_1[t] = s_1[t-1] + x_1[t]`

The edge `add_single → add_single` is the self-loop that returns the accumulated state. On the
first time step, `initial_state` returns `{'s_1': None}`, which the node treats as 0.

In [ ]:
class Add1(Node):
    def __init__(self, name):
        def add(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            v = s_1 + X['x_1']
            return {'s_1':  v}
        super().__init__(
            node=add,
            inputs={'x_1': np.ones((1, 3)), 's_1': np.ones((1,))},
            outputs={'s_1': np.ones((1,))},
            name=name,
            recurrent=True
        )

    def initial_state(self, X):
        return {'s_1': None}

htg_single = HierarchalTensorGraph(name='parent')
add_single = Add1('add_1')

htg_single.add_edge('input', add_single, rollout_axis=1, features=['x_1'])
htg_single.add_edge(add_single, add_single, rollout_axis=1)
htg_single.add_edge(add_single, 'output')

In [ ]:
htg_single.draw()

## Example 3: Three Mutually-Coupled Recurrent Nodes (All-to-All)

This example extends bidirectional coupling to three nodes with **all-to-all state sharing**:
every node receives the previous state from every other node as well as its own.

- `Add1`: `s_1[t] = s_1[t-1] + s_2[t-1] + s_3[t-1] + x_1[t]`
- `Add2`: `s_2[t] = s_1[t-1] + s_2[t-1] + s_3[t-1] + x_2[t]`
- `Add3`: `s_3[t] = s_1[t-1] + s_2[t-1] + s_3[t-1] + x_3[t]`

This fully-coupled topology represents ESM sub-systems where every component exchanges state
at each model time step (e.g. ocean, atmosphere, and land surface interacting simultaneously).
All six cross-edges plus three self-loops are required to express this structure.

In [ ]:
class Add1(Node):
    def __init__(self, name):
        def add(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            s_2 = X['s_2'] if X['s_2'] is not None else 0
            s_3 = X['s_3'] if X['s_3'] is not None else 0
            v = s_1 + s_2 + s_3 + X['x_1']
            return {'s_1':  v}
        super().__init__(
            node=add,
            inputs={'x_1': np.ones((1, 3)), 's_3': np.ones((1,)), 's_2': np.ones((1,)), 's_1': np.ones((1,))},
            outputs={'s_1': np.ones((1,))},
            name=name,
            recurrent=True
        )

    def initial_state(self, X):
        return {'s_1': None}


class Add2(Node):
    def __init__(self, name):
        def add(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            s_2 = X['s_2'] if X['s_2'] is not None else 0
            s_3 = X['s_3'] if X['s_3'] is not None else 0
            v = s_1 + s_2 + s_3 + X['x_2']
            return {'s_2':  v}
        super().__init__(
            node=add,
            inputs={'x_2': np.ones((1, 3)), 's_3': np.ones((1,)), 's_2': np.ones((1,)), 's_1': np.ones((1,))},
            outputs={'s_2': np.ones((1,))},
            name=name,
            recurrent=True
        )

    def initial_state(self, X):
        return {'s_2': None}


class Add3(Node):
    def __init__(self, name):
        def add(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            s_2 = X['s_2'] if X['s_2'] is not None else 0
            s_3 = X['s_3'] if X['s_3'] is not None else 0
            v = s_1 + s_2 + s_3 + X['x_3']
            return {'s_3':  v}
        super().__init__(
            node=add,
            inputs={'x_3': np.ones((1, 3)), 's_3': np.ones((1,)), 's_2': np.ones((1,)), 's_1': np.ones((1,))},
            outputs={'s_3': np.ones((1,))},
            name=name,
            recurrent=True
        )

    def initial_state(self, X):
        return {'s_3': None}

In [ ]:
htg = HierarchalTensorGraph(name='parent')
add_1 = Add1('add_1')
add_2 = Add2('add_2')
add_3 = Add3('add_3')

htg.add_edge('input', add_1, rollout_axis=1, features=['x_1'])
htg.add_edge('input', add_2, rollout_axis=1, features=['x_2'])
htg.add_edge('input', add_3, rollout_axis=1, features=['x_3'])
htg.add_edge(add_1, add_2, rollout_axis=1)
htg.add_edge(add_2, add_1, rollout_axis=1)
htg.add_edge(add_2, add_3, rollout_axis=1)
htg.add_edge(add_3, add_2, rollout_axis=1)
htg.add_edge(add_3, add_1, rollout_axis=1)
htg.add_edge(add_1, add_3, rollout_axis=1)
htg.add_edge(add_1, add_1, rollout_axis=1)
htg.add_edge(add_2, add_2, rollout_axis=1)
htg.add_edge(add_3, add_3, rollout_axis=1)
htg.add_edge(add_1, 'output')
htg.add_edge(add_2, 'output')
htg.add_edge(add_3, 'output')

In [ ]:
htg.draw()

## Example 4: Unidirectional Chain Coupling

In this example the coupling is strictly one-way.

#### Node Definitions

`Add1` accumulates the external sequence input with its own previous state.
`Add2` accumulates the state forwarded by `Add1` with its own previous state.
Neither node shares state in both directions — the coupling is strictly one-way.

In [ ]:
class Add1(Node):
    def __init__(self, name):
        def add(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            v = X['x_1'] + s_1
            return {'s_1':  v}
        super().__init__(
            node=add,
            inputs={'x_1': None, 's_1': None},
            outputs={'s_1': None},
            name=name,
            recurrent=True
        )

    def initial_state(self, X):
        return {'s_1': None}


class Add2(Node):
    def __init__(self, name):
        def add(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            s_2 = X['s_2'] if X['s_2'] is not None else 0
            v = s_1 + s_2
            return {'s_2':  v}
        super().__init__(
            node=add,
            inputs={'s_1': None, 's_2': None},
            outputs={'s_2': None},
            name=name,
            recurrent=True
        )

    def initial_state(self, X):
        return {'s_2': None}

### Build and Run the Graph

Note that `add_2` has no edge from `'input'` — it receives no external sequence. Its only inputs are the state forwarded by `add_1` and its own accumulated state. `draw()` renders the graph so you can verify the unidirectional coupling visually.

In [ ]:
htg = HierarchalTensorGraph(name='parent')
add_1 = Add1('add_1')
add_2 = Add2('add_2')

htg.add_edge('input', add_1, rollout_axis=1, features=['x_1'])
htg.add_edge(add_1, add_2, rollout_axis=1)
htg.add_edge(add_1, add_1, rollout_axis=1)
htg.add_edge(add_2, add_2, rollout_axis=1)
htg.add_edge(add_2, 'output')

htg.draw()

## Example 5: Recurrent HTGs with TensorFlow

The next two cases use **tf.TensorSpec** to declare explicit shapes and dtypes,
which is required when nodes will be compiled with `tf.function` or when you need strict
shape checking during graph construction.

We also introduce the **return_seq** parameter:

| `return_seq` | Effect |
|---|---|
| `False` (default) | The node outputs only the **final state** after processing the full sequence |
| `True` | The node outputs the **full sequence** of states, one per time step |

Use `return_seq=True` when downstream nodes or the HTG output need intermediate states,
not just the result after the last time step.

What follows demonstrates two cases:
1. A single `Add1` node (self-loop, `return_seq=False`) — returns only the last state
2. A coupled `Add1`/`Add2` system where `Add2` has `return_seq=True` — returns all states

## Node Definitions

`Add1` (`return_seq=False`) and `Add2` (`return_seq=True`) both use `tf.TensorSpec` for
their I/O declarations. The `shape` in `TensorSpec` reflects `(batch, features)` for states
and `(batch, time, features)` for sequence inputs; CREST slices along the rollout axis
automatically during execution.

In [ ]:
class Add1(Node):
    def __init__(self):
        def add1(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            v = s_1 + X['x']
            return {'s_1': v}
        super().__init__(
            node=add1,
            inputs={'x': tf.TensorSpec(shape=(1, 3)), 's_1': tf.TensorSpec(shape=(1,))},
            outputs={'s_1': tf.TensorSpec(shape=(1,))},
            name='add1',
            recurrent=True,
            return_seq=False
        )

    def initial_state(self, X):
        return {'s_1': None}


class Add2(Node):
    def __init__(self):
        def add2(X):
            s_1 = X['s_1'] if X['s_1'] is not None else 0
            s_2 = X['s_2'] if X['s_2'] is not None else 0
            v = s_1 + s_2 + X['x']
            return {'s_2': v}
        super().__init__(
            node=add2,
            inputs={'x': tf.TensorSpec(shape=(1, 3)), 's_2': tf.TensorSpec(shape=(1,)), 's_1': tf.TensorSpec(shape=(1,))},
            outputs={'s_2': tf.TensorSpec(shape=(1,))},
            name='add2',
            recurrent=True,
            return_seq=True
        )

    def initial_state(self, X):
        return {'s_2': None}

## Case 1: Single `Add1` Node (`return_seq=False`)

`Add1` computes a running cumulative sum: `s_1[t] = s_1[t-1] + x[t]`. With `return_seq=False`
only the final accumulated value (after the last time step) is returned in the output dict.

The edges here omit `rollout_axis` explicitly — CREST automatically applies `rollout_axis=1`
to any edge targeting a recurrent node.

`htg_test.summary` shows the inferred input/output tensor spec for the whole graph.
The call passes a tensor of shape `(2, 3)`: batch size 2, sequence length 3.

In [ ]:
add1_test = Add1()

htg_test = HierarchalTensorGraph(name='parent')
htg_test.add_edge('input', add1_test)
htg_test.add_edge(add1_test, add1_test)
htg_test.add_edge(add1_test, 'output')

In [ ]:
htg_test.summary

In [ ]:
htg_test({'x': tf.constant([[1., 2, 3], [4, 5, 6]], dtype=tf.float32)})

In [ ]:
htg_test.draw()

## Case 2: Coupled `Add1`/`Add2` with `return_seq=True`

Both nodes receive `x` from the external input and share state bidirectionally:

- `add1 → add2`: `add1`'s `s_1` becomes `add2`'s `s_1` input at the next step.
- `add2 → add1`: `add2`'s `s_2` is selected (`features='s_2'`) and renamed to `s_1`
  (`rename={'s_2': 's_1'}`) before being passed to `add1`.
- `add2 → add2`: self-loop carrying `add2`'s own state forward.

Because `Add2` has `return_seq=True`, its output `s_2` contains the **full sequence**
of accumulated states (one value per time step), not just the final value.
`Add1` still returns only its final `s_1` state.

In [ ]:
add1 = Add1()
add2 = Add2()

htg = HierarchalTensorGraph(name='parent')
htg.add_edge('input', add1, rollout_axis=1)
htg.add_edge('input', add2, rollout_axis=1, features='x')
htg.add_edge(add1, add2, rollout_axis=1)
htg.add_edge(add2, add1, rollout_axis=1, features='s_2', rename={'s_2': 's_1'})
htg.add_edge(add2, add2, rollout_axis=1)
htg.add_edge(add1, 'output')
htg.add_edge(add2, 'output')

In [ ]:
htg.draw()